In [3]:
import pandas as pd
import numpy as np

np.random.seed(17)
n = 104
t = np.arange(n)

# Tren: naik sekitar 0.45 pemesanan per minggu
trend = 18 + 0.45 * t

# Musiman tahunan: puncak di akhir tahun (November-Desember), fase berbeda dari materi
tahunan = 14 * np.sin(2 * np.pi * t / 52 + 2.2)

# Komponen setengah tahunan: puncak kecil di Maret dan Agustus
semester = 5 * np.sin(4 * np.pi * t / 52 - 0.3)

# Noise lebih kecil dari materi (standar deviasi 2.5)
noise = np.random.normal(0, 2.5, n)

pemesanan = np.round(trend + tahunan + semester + noise).clip(min=0)

df = pd.DataFrame({
    'minggu': np.arange(1, n + 1),
    'tanggal': pd.date_range('2024-01-01', periods=n, freq='W'),
    'pemesanan': pemesanan
})
df.to_csv('fotografi_pemesanan.csv', index=False)

print(df.head())
print("\nStatistik deskriptif:")
print(df['pemesanan'].describe().round(1))

   minggu    tanggal  pemesanan
0       1 2024-01-07       29.0
1       2 2024-01-14       24.0
2       3 2024-01-21       30.0
3       4 2024-01-28       32.0
4       5 2024-02-04       32.0

Statistik deskriptif:
count    104.0
mean      41.5
std       18.7
min        6.0
25%       29.8
50%       42.5
75%       52.0
max       77.0
Name: pemesanan, dtype: float64


In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.deterministic import DeterministicProcess
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBRegressor

df = pd.read_csv('fotografi_pemesanan.csv', parse_dates=['tanggal'], index_col='tanggal')
df = df.asfreq('W')

n_test = 20
train = df.iloc[:-n_test]
test = df.iloc[-n_test:]

# Model tren
dp = DeterministicProcess(index=train.index, constant=True, order=1, drop=True)
X_tren_train = dp.in_sample()
X_tren_test = dp.out_of_sample(steps=n_test)

model_tren = LinearRegression()
model_tren.fit(X_tren_train, train['pemesanan'])
pred_tren_train = model_tren.predict(X_tren_train)
pred_tren_test = model_tren.predict(X_tren_test)

print("Koefisien model tren:")
for nama, k in zip(X_tren_train.columns, model_tren.coef_):
    print(f"  {nama}: {k:.3f}")
print(f"Intercept: {model_tren.intercept_:.2f}")

# Residual
residual_train = train['pemesanan'] - pred_tren_train
residual_test = test['pemesanan'] - pred_tren_test
print("\nStatistik residual data latih:")
print(residual_train.describe().round(2))

# Fitur musiman
def buat_fitur_musiman(idx):
    fitur = pd.DataFrame(index=idx)
    fitur['bulan'] = idx.month
    fitur['minggu_ke'] = idx.isocalendar().week.astype(int)
    return fitur

X_musim_train = buat_fitur_musiman(train.index)
X_musim_test = buat_fitur_musiman(test.index)

# Evaluasi sebelum tuning
model_sisa = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
model_sisa.fit(X_musim_train, residual_train)
pred_hybrid_test = pred_tren_test + model_sisa.predict(X_musim_test)

mae_tren = mean_absolute_error(test['pemesanan'], pred_tren_test)
mae_hybrid = mean_absolute_error(test['pemesanan'], pred_hybrid_test)
print(f"\nMAE tren saja: {mae_tren:.2f}")
print(f"MAE hybrid: {mae_hybrid:.2f}")
print(f"Perbaikan: {((mae_tren - mae_hybrid) / mae_tren * 100):.1f}%")

# Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5],
}
tscv = TimeSeriesSplit(n_splits=4)
pencari = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=tscv,
                       scoring='neg_mean_absolute_error', n_jobs=-1)
pencari.fit(X_musim_train, residual_train)
print("\nParameter terbaik:", pencari.best_params_)

model_sisa = pencari.best_estimator_
pred_hybrid_test = pred_tren_test + model_sisa.predict(X_musim_test)
print(f"MAE hybrid setelah tuning: {mean_absolute_error(test['pemesanan'], pred_hybrid_test):.2f}")

# Peramalan 12 minggu
n_forecast = 12
X_tren_forecast = dp.out_of_sample(steps=n_test + n_forecast).iloc[-n_forecast:]
pred_tren_forecast = model_tren.predict(X_tren_forecast)

idx_forecast = pd.date_range(df.index[-1] + pd.Timedelta(days=7), periods=n_forecast, freq='W')
X_musim_forecast = buat_fitur_musiman(idx_forecast)
pred_hybrid_forecast = pred_tren_forecast + model_sisa.predict(X_musim_forecast)

deviasi = pred_hybrid_forecast - pred_tren_forecast
hasil = pd.DataFrame({
    'tanggal': idx_forecast,
    'prediksi': pred_hybrid_forecast.round(1),
    'tren': pred_tren_forecast.round(1),
    'deviasi': deviasi.round(1)
})
print("\nPeramalan 12 minggu ke depan:")
print(hasil)

print("\nRekomendasi penjadwalan fotografer:")
for i, row in hasil.iterrows():
    if row['deviasi'] < -2:
        label = "fotografer dikurangi"
    elif row['deviasi'] > 2:
        label = "fotografer ekstra"
    else:
        label = "fotografer normal"
    print(f"  {row['tanggal'].date()}: prediksi {row['prediksi']:.1f}, "
          f"tren {row['tren']:.1f}, deviasi {row['deviasi']:+.1f} -> {label}")

Koefisien model tren:
  const: 0.000
  trend: 0.363
Intercept: 19.51

Statistik residual data latih:
count    84.00
mean     -0.00
std      10.90
min     -21.14
25%      -9.34
50%       2.18
75%       9.43
max      20.24
Name: pemesanan, dtype: float64

MAE tren saja: 15.43
MAE hybrid: 6.66
Perbaikan: 56.8%

Parameter terbaik: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200}
MAE hybrid setelah tuning: 7.69

Peramalan 12 minggu ke depan:
      tanggal  prediksi  tren  deviasi
0  2026-01-04      66.5  57.6      8.8
1  2026-01-11      65.1  58.0      7.1
2  2026-01-18      68.2  58.4      9.8
3  2026-01-25      68.5  58.7      9.8
4  2026-02-01      68.9  59.1      9.8
5  2026-02-08      69.3  59.4      9.8
6  2026-02-15      66.1  59.8      6.3
7  2026-02-22      65.1  60.2      5.0
8  2026-03-01      65.4  60.5      4.9
9  2026-03-08      64.0  60.9      3.1
10 2026-03-15      64.4  61.3      3.2
11 2026-03-22      58.9  61.6     -2.7

Rekomendasi penjadwalan fotografer:
  2